# WLASL300 Sign Language Recognition — Training Notebook

End-to-end training pipeline on Google Colab.

**Prerequisites:**
1. Enable GPU: `Runtime → Change runtime type → T4 GPU`
2. Mount Google Drive (cell below) — used to persist checkpoints across sessions.
3. Upload the Kaggle credentials JSON or set `KAGGLE_USERNAME` / `KAGGLE_KEY` secrets.

**Sections:**
1. Environment setup
2. Mount Google Drive
3. Download dataset
4. Build annotations
5. Training
6. Evaluation and plots

## 1. Environment setup

In [1]:
# Verify GPU is available
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

CUDA available: True
GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB


In [2]:
# Clone the project repository
# Replace with your actual repository URL
!git clone https://github.com/jibran/wlasl300-sign-language.git
%cd wlasl300-sign-language

Cloning into 'wlasl300-sign-language'...
remote: Enumerating objects: 109, done.
remote: Counting objects: 100% (109/109), done.
remote: Compressing objects: 100% (74/74), done.
remote: Total 109 (delta 34), reused 104 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (109/109), 999.06 KiB | 11.22 MiB/s, done.
Resolving deltas: 100% (34/34), done.
/content/wlasl300-sign-language


In [3]:
!git pull origin main

From https://github.com/jibran/wlasl300-sign-language
 * branch            main       -> FETCH_HEAD
Already up to date.


In [4]:
# Install dependencies (uv is not available on Colab — use pip)
!pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.9/91.9 kB 9.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.5/117.5 kB 12.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 5.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 5.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.7/132.7 kB 17.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 7.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 78.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.7/107.7 kB 15.0 MB/s eta 0:0

In [5]:
# Add project root to sys.path so all local imports resolve
import sys, os

PROJECT_ROOT = os.path.abspath(".")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.environ["PYTHONPATH"] = PROJECT_ROOT
print(f"Project root: {PROJECT_ROOT}")

Project root: /content/wlasl300-sign-language


## 2. Mount Google Drive

In [6]:
from google.colab import drive

drive.mount("/content/drive")

# Set this to your Drive folder for persisting data and checkpoints
DRIVE_DIR = "/content/drive/MyDrive/wlasl300"

import os

os.makedirs(f"{DRIVE_DIR}/trained_models/best", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/trained_models/latest", exist_ok=True)

# Symlink Drive dirs into the project so paths in config.yaml work
!ln -sfn {DRIVE_DIR}/trained_models/best trained_models/best
!ln -sfn {DRIVE_DIR}/trained_models/latest trained_models/latest
print("Drive mounted and symlinks created.")

Mounted at /content/drive
Drive mounted and symlinks created.


## 3. Download dataset from Kaggle

In [7]:
# Load Kaggle credentials from Colab secrets
# Add KAGGLE_USERNAME and KAGGLE_KEY in:
#   Colab sidebar → Secrets (key icon) → + Add new secret
from google.colab import userdata
import os

os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

!mkdir -p ~/.kaggle
!echo '{"username":"'$KAGGLE_USERNAME'","key":"'$KAGGLE_KEY'"}' > ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# Weights & Biases (W&B) — optional experiment tracker
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
os.environ["WANDB_PROJECT"] = userdata.get("WANDB_PROJECT")
os.environ["WANDB_ENTITY"] = userdata.get("WANDB_ENTITY")

In [8]:
print("Downloading WLASL300 dataset (~2 GB) ...")
!kaggle datasets download vodinhnhattruong/dataset-wlasl300 \
    --unzip --path dataset/
print("Dataset downloaded.")
!ls -al dataset/dataset_wlasl300/
!ls -al dataset/raw/
!rm -rf dataset/raw/
!mv dataset/dataset_wlasl300 dataset/raw
!ls -al dataset/raw/

Dataset URL: https://www.kaggle.com/datasets/vodinhnhattruong/dataset-wlasl300
License(s): unknown
100% 1.93G/1.93G [02:15<00:00, 15.4MB/s]

Dataset downloaded.
total 24
drwxr-xr-x   4 root root 4096 Mar 18 11:56 .
drwxr-xr-x   8 root root 4096 Mar 18 11:57 ..
-rw-r--r--   1 root root 2478 Mar 18 11:56 folder2label_int.txt
-rw-r--r--   1 root root 3226 Mar 18 11:56 folder2label_str.txt
drwxr-xr-x   5 root root 4096 Mar 18 11:57 preprocessing
drwxr-xr-x 302 root root 4096 Mar 18 11:56 WLASL300
total 8
drwxr-xr-x 2 root root 4096 Mar 18 11:52 .
drwxr-xr-x 8 root root 4096 Mar 18 11:57 ..
-rw-r--r-- 1 root root    0 Mar 18 11:52 .gitkeep
total 24
drwxr-xr-x   4 root root 4096 Mar 18 11:56 .
drwxr-xr-x   7 root root 4096 Mar 18 11:57 ..
-rw-r--r--   1 root root 2478 Mar 18 11:56 folder2label_int.txt
-rw-r--r--   1 root root 3226 Mar 18 11:56 folder2label_str.txt
drwxr-xr-x   5 root root 4096 Mar 18 11:57 preprocessing
drwxr-xr-x 302 root root 4096 Mar 18 11:56 WLASL300


## 4. Build annotations and Word2Vec embeddings

In [ ]:
import os

DRIVE_W2V = f"{DRIVE_DIR}/trained_models/embeddings/GoogleNews-vectors-negative300.bin"
OUT_BIN   = "trained_models/embeddings/GoogleNews-vectors-negative300.bin"
OUT_GZ    = OUT_BIN + ".gz"
# Option 1: Kaggle dataset (most reliable — requires Kaggle credentials)
!kaggle datasets download -d leadbest/googlenewsvectorsnegative300 \
    -p trained_models/embeddings/ --unzip

if not os.path.exists(OUT_BIN):
    # Option 2: gdown from Google Drive
    !pip install -q gdown
    !gdown "0B7XkCwpI5KDYNlNUTTlSS21pQmM" -O {OUT_GZ}
    !gunzip -f {OUT_GZ}

if os.path.exists(OUT_BIN):
    # Persist to Drive so future sessions skip the download
    os.makedirs(os.path.dirname(DRIVE_W2V), exist_ok=True)
    !cp {OUT_BIN} {DRIVE_W2V}
    size_gb = os.path.getsize(OUT_BIN) / 1e9
    print(f"Word2Vec binary ready and saved to Drive ({size_gb:.1f} GB)")
else:
    print("ERROR: could not download Word2Vec binary. Try opening the link manually:")
    print("  https://drive.google.com/uc?id=0B7XkCwpI5KDYNlNUTTlSS21pQmM")
    print(f"Then upload to: {DRIVE_W2V}")


Dataset URL: https://www.kaggle.com/datasets/leadbest/googlenewsvectorsnegative300
License(s): other
100% 3.17G/3.17G [03:35<00:00, 15.9MB/s]

Word2Vec binary ready and saved to Drive (3.6 GB)


In [10]:
!PYTHONPATH=/content/wlasl300-sign-language python dataset/annotations/build_annotations.py \
    --preprocessing_dir  dataset/raw/preprocessing \
    --folder2label       dataset/raw/folder2label_str.txt \
    --word2vec_bin       trained_models/embeddings/GoogleNews-vectors-negative300.bin \
    --out_dir            dataset/annotations

12:01:49  INFO      WLASL300 annotation pipeline starting
12:01:49  INFO        preprocessing_dir : dataset/raw/preprocessing
12:01:49  INFO        wlasl_dir         : WLASL300
12:01:49  INFO        folder2label      : dataset/raw/folder2label_str.txt
12:01:49  INFO        word2vec_bin      : trained_models/embeddings/GoogleNews-vectors-negative300.bin
12:01:49  INFO        out_dir           : dataset/annotations
12:01:49  INFO        num_frames        : 16
12:01:49  INFO      Loaded 300 class labels from dataset/raw/folder2label_str.txt
12:01:51  INFO      Clip discovery: 5113 complete, 0 incomplete
12:01:51  INFO        train  : 3270 clips
12:01:51  INFO        val    : 819 clips
12:01:51  INFO        test   : 1024 clips
12:01:51  INFO      Vocabulary: 300 classes
12:01:51  INFO      Splits — train: 3270  val: 819  test: 1024
12:01:54  INFO      Loading Word2Vec from trained_models/embeddings/GoogleNews-vectors-negative300.bin (~30 s) …
12:01:54  INFO      loading projection weights 

## 5. Training

In [ ]:
# Start training (uses config/config.yaml)
# Reduce batch_size if you hit CUDA OOM
!PYTHONPATH=/content/wlasl300-sign-language python train/train.py \
    --config config/config.yaml

12:02:28  INFO      __main__  Random seed set to 42
12:02:28  INFO      __main__  Using CUDA device: NVIDIA A100-SXM4-40GB (42.4 GB)
12:02:28  INFO      __main__  Building dataloaders …
12:02:28  INFO      dataset.data.wlasl_dataset  WLASL300Dataset  split='train'  samples=3270  classes=300
12:02:28  INFO      dataset.data.wlasl_dataset  WLASL300Dataset  split='val'  samples=819  classes=300
12:02:28  INFO      dataset.data.wlasl_dataset  WLASL300Dataset  split='test'  samples=1024  classes=300
12:02:28  INFO      dataset.data.wlasl_dataset  DataLoaders — train=408  val=52  test=64 batches
Downloading: "https://dl.fbaipublicfiles.com/pytorchvideo/model_zoo/kinetics/I3D_8x8_R50.pyth" to /root/.cache/torch/hub/checkpoints/I3D_8x8_R50.pyth
100% 214M/214M [00:01<00:00, 188MB/s]
12:02:31  WARNING   models.i3d_backbone  I3DBackbone: config backbone_output_dim=1024 does not match actual backbone output 2048 — overriding with 2048. Update config.yaml: model.backbone_output_dim: 2048
12:02:31  

In [ ]:
# To resume from a checkpoint:
# !python train/train.py \
#     --config config/config.yaml \
#     --resume trained_models/latest/checkpoint_epoch010.pt

## 6. Evaluation and visualisation

In [ ]:
from IPython.display import Image, display
import os

plot_dir = "logs/plots"
for fname in ["loss_curves.png", "accuracy_curves.png", "throughput.png"]:
    path = os.path.join(plot_dir, fname)
    if os.path.exists(path):
        print(f"--- {fname} ---")
        display(Image(path))

In [ ]:
# Run inference on a sample video
# Replace with the path to a real video file
SAMPLE_VIDEO = "WLASL300/0/00412.mp4"

!PYTHONPATH=/content/wlasl300-sign-language python inference/inference.py \
    --checkpoint trained_models/best/checkpoint.pt \
    --video      {SAMPLE_VIDEO} \
    --top_k      5 \
    --verbose